# Multimodal

`Use a vision capable LLM to read and interpret it.`

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Encode the image and send to the vision model

In [4]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [ ]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}, # * f"data:image/png;base64,{image_b64}" standard image
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

**Extracted Test Results with Flagged Abnormal Values**

The following test results have been extracted from the blood work report:

### Complete Blood Count (CBC)

*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) **- Normal**
*   **Hematocrit:** 44% (Normal: 41-53) **- Normal**
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0) **- Normal**
*   **Platelets:** 220 x10^3/uL (Normal: 150-400) **- Normal**

### Lipid Panel

*   **Total Cholesterol:** 238 mg/dL (Normal: <200) **- Elevated**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) **- Elevated**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) **- Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) **- Elevated**

### Metabolic Panel

*   **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) **- Normal**
*   **HbA1c:** 5.3% (Normal: <5.7) **- Normal**
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7-1.3) **- Normal**
*   **eGFR:** 82 mL/min (Normal: >60) **- Normal**

### Liver Function

*   No results provided.

The flagged abnormal values are:


## Suggest Diet Plan Agent

`The agent will read the blood work image, categorizes the condition, then calls the diet tool.`

In [6]:
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

In [7]:
@tool
def get_diet_recommendation(health_condition: str) -> dict:
    """Given a health condition, return a diet plan. Condiino must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(health_condition, diet_plans["normal"])

In [12]:
SYSTEM_PROMPT = """
    You are a helpful medical and nutrition assistant.
    For the input blood work image, extract the numbers and the normal range, then categorize
    the condition as one of: normal, high_cholesterol, high_sugar.
    Then call he appropriate tool to retrieve and present the diet plan
"""


diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
    # checkpointer=InMemorySaver()    
)

In [14]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}, # * f"data:image/png;base64,{image_b64}" standard image
        {"type": "text",      "text": "Analyze this blood work repor and suggest a diet plan."}
    ])]
})

print(result["messages"][-1].content)

Based on the blood work report, the patient has high cholesterol. A suitable diet plan for someone with high cholesterol includes eating fruits, vegetables, whole grains, and lean protein, while avoiding red meat, fried food, full-fat dairy, and processed snacks.


# Evals

```text
                                   Evals
                                     │
        ┌────────────────────────────┼────────────────────────────┐
        │                            │                            │
        ▼                            ▼                            ▼

┌─────────────────────┐     ┌─────────────────────┐     ┌─────────────────────┐
│   Functional Eval   │     │      Cost Eval      │     │    Safety Eval      │
└─────────────────────┘     └─────────────────────┘     └─────────────────────┘
        │                            │                            │
        ▼                            ▼                            ▼

  ┌──────────────────────┐      ┌──────────────────────┐      ┌──────────────────────┐
  │ 1. Is answer correct?│      │ 1. Tokens            │      │ 1. Toxic output      │
  └──────────────────────┘      └──────────────────────┘      └──────────────────────┘
               │                           │                            │
               ▼                           ▼                            ▼
  ┌─────────────────────────────┐ ┌──────────────────────┐ ┌──────────────────────────┐
  │ 2. Faithfulness,            │ │ 2. Latency           │ │ 2. PII leak, jailbreak   │
  │    hallucination            │ └──────────────────────┘ └──────────────────────────┘
  └─────────────────────────────┘
```